# 08 · Validación final, sesgos y robustez (Fases 11 y 13)

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_columns", 30)
from src.config import *
df = pd.read_parquet(PROCESSED / "emovi_movilidad.parquet")
q1 = df[df.sigue_q1.notna()]
from src.models import train as T
tr, te = T.split(df)
tabla, modelos = T.compare(tr, te, T.NUM, T.CAT)
elegido, mejor_auc = T.choose(tabla)
modelo = modelos[elegido]
from src.features.indices import weighted_quintile

## 1. Robustez: ¿el hallazgo principal depende de cómo se mide?

In [2]:
w = df.factor
def tasa(qo, qd, m=None):
    m = (qo == 1) if m is None else m
    return np.average(qd[m] == 1, weights=w[m]) * 100
qo_all = weighted_quintile(df.indice_origen, w); qd_all = weighted_quintile(df.indice_destino, w)
qi = pd.Series(0, index=df.index)
for _, g in df.groupby("cohorte"): qi[g.index] = weighted_quintile(g.ingreso_pc, g.factor)
from src.evaluation import extra as X
ic = X.cboot(q1)
rob = pd.DataFrame({"Definición": ["Principal: índice de bienes, quintiles por cohorte", "Quintiles sin separar por cohorte", "Destino medido con ingreso per cápita"],
                    "% sigue en Q1": [tasa(df.q_origen, df.q_destino), tasa(qo_all, qd_all), tasa(df.q_origen, qi)]}).round(1)
rob.to_csv(TABLES / "robustez.csv", index=False); display(rob)
print(f"Definición principal: {ic['%']:.1f}% — IC 95% por conglomerados (UPM): {ic['IC 95% inf']:.1f}% – {ic['IC 95% sup']:.1f}% | n = {ic['n']}, n efectivo = {ic['n efectivo']:.0f}")
pd.DataFrame([ic]).round(1).to_csv(TABLES / "ic_persistencia.csv", index=False)

,Definición,% sigue en Q1
0,"Principal: índice de bienes, quintiles por coh...",51.1
1,Quintiles sin separar por cohorte,48.9
2,Destino medido con ingreso per cápita,46.6


Definición principal: 51.1% — IC 95% por conglomerados (UPM): 46.3% – 55.4% | n = 2674, n efectivo = 815


**Conclusión:** en todas las definiciones, entre 47% y 51% de quienes crecen en el 20% de hogares con menos bienes siguen ahí. El IC 95% que respeta el diseño muestral es **46–55%** (corrección C-02: el IC anterior, 48–55%, ignoraba los conglomerados). El hallazgo es robusto. El orden de magnitud es consistente con lo que reporta el CEEY (la cifra exacta se debe verificar en su informe; ver `docs/report/referencias.md`).

## 2. Desempeño por subgrupo (sesgo del modelo)
Se usan **predicciones fuera de muestra** (CV de 5 particiones) para las 2,674 personas de Q1, con IC 95% bootstrap de la brecha real − predicha (corrección C-01).

In [3]:
sub = T.subgroup_performance(modelo, df, T.NUM, T.CAT)
sub.to_csv(TABLES / "desempeno_subgrupos.csv", index=False); display(sub)

,Variable,Grupo,n,Tasa real sigue Q1,Tasa predicha,Brecha (pts),IC 95% inf,IC 95% sup,AUC
0,mujer,Hombre,1047,0.506,0.506,-0.020,-2.723,2.850,0.696
1,mujer,Mujer,1627,0.526,0.525,0.036,-2.371,2.357,0.702
2,tono_piel,Claro (H-K),417,0.451,0.451,-0.020,-4.333,4.912,0.717
3,tono_piel,Medio (F-G),1679,0.525,0.525,0.031,-2.030,2.520,0.693
4,tono_piel,Oscuro (A-E),578,0.545,0.545,-0.014,-3.992,3.930,0.695
5,region_14,Centro,632,0.486,0.485,0.123,-3.798,3.768,0.637
6,region_14,Centro-norte,243,0.313,0.317,-0.435,-6.839,5.568,0.564
7,region_14,Norte,198,0.328,0.327,0.105,-6.298,6.722,0.549
8,region_14,Norte-occidente,242,0.310,0.314,-0.412,-5.551,5.237,0.624
9,region_14,Sur,1355,0.635,0.634,0.173,-2.379,2.933,0.660


**Resultado:** el modelo está bien calibrado por sexo, lengua, región, localidad y **tono de piel**: todas las brechas tienen IC que incluyen el 0. El AUC dentro de cada sexo, tono o lengua es similar al global (≈0.65–0.72). Dentro de cada **región** baja (≈0.55–0.66) porque la región, que es la señal más fuerte del modelo, ya no varía dentro del grupo.

**Hallazgo retirado (auditoría H-01, decisión D25):** la versión anterior reportaba que el modelo "subestimaba en ≈11 puntos la persistencia de las personas de tono oscuro". Ese resultado venía de **una sola partición de prueba** (114 personas) en la que, por azar, la tasa real de ese grupo fue 61% frente a 53% en entrenamiento. Con predicciones fuera de muestra para todas las personas, la brecha es ≈0. Se conserva en la bitácora como ejemplo de por qué no se debe evaluar subgrupos con una sola partición.

## 3. Red team (Fase 11): riesgos clasificados
| # | Riesgo | Nivel | Evidencia | Acción |
|---|---|---|---|---|
| 1 | **Causalidad injustificada** en los escenarios | **ALTO** | Escolaridad y empleo se miden al mismo tiempo que el destino | Lenguaje condicional obligatorio; se rechazó el escenario de choque adverso |
| 2 | **Evaluar subgrupos con una sola partición** | **ALTO** (detectado en la auditoría) | Produjo una brecha espuria de ≈11 pts por tono | **Corregido:** predicciones fuera de muestra con IC; hallazgo retirado (D25). El modelo no debe usarse para focalizar personas por su AUC moderado |
| 3 | **Decisiones tomadas después de ver resultados** (D16, D17) | **MEDIO** | Agrupar la escolaridad de los padres y quitar un escenario | Se justificaron por tamaño de muestra, no por resultados; se dejaron registradas |
| 4 | **Efecto de cohorte como artefacto** | **MEDIO** | La auditoría **refutó** que se deba a los bienes modernos (celular, internet, etc.): sin ellos, la persistencia por cohorte es igual (59.4% vs 59.6% en 25-34) | Queda abierto el ciclo de vida; no se afirma que la movilidad empeoró |
| 5 | **Sesgo de memoria** (origen retrospectivo) | MEDIO | — | Validación con la escalera subjetiva (r ≈ 0.5); se usan bienes, no ingreso |
| 6 | **Sesgo de supervivencia** | MEDIO | La muestra va de 25 a 64 años: quienes murieron o emigraron no están | Limitación declarada; puede subestimar la persistencia si la pobreza aumenta la mortalidad o la emigración |
| 7 | **Falacia ecológica y representatividad estatal** | **ALTO** | La EMOVI no es representativa por estado (n efectivo ≈ 31); pobreza 2022 vs origen de décadas atrás | Se presenta como **exploratorio**, con el n efectivo visible |
| 8 | **Missingness no aleatoria** | BAJO | El padre ausente se asocia con más persistencia | Categoría propia, sin imputación |
| 9 | **Leakage** | BAJO | Revisado: sin variables de destino; imputación dentro del pipeline | — |
| 10 | **Desbalance** | BAJO | 52/48 | Sin efecto (`class_weight` no cambia nada) |
| 11 | **Confusión entre movilidad absoluta y relativa** | BAJO | Se mide movilidad **relativa** (quintiles) | Declarado: que alguien "siga en Q1" no significa que viva igual que sus padres |
| 12 | **Confusión entre educación y oportunidades laborales** | MEDIO | La educación y el empleo formal están correlacionados | Se separaron en los escenarios de la Fase 10 |
| 13 | **Promedios que ocultan heterogeneidad** | MEDIO | Las diferencias por tono podrían variar por región, pero no es significativo | Reformulado como pregunta abierta (C-05) |
| 15 | **Incertidumbre subestimada** | **ALTO** (detectado en la auditoría) | Bootstrap sin conglomerados; n efectivo de Q1 ≈ 815 | **Corregido:** IC por conglomerados en todas las tasas (C-03) |
| 16 | **Variables medidas hoy** (lengua, tono autopercibido, autoadscripción) | MEDIO | Pueden reflejar el destino, no solo el origen | El equipo decidió conservarlas en el modelo; se interpretan con esa restricción |
| 14 | **Visualizaciones engañosas** | BAJO | Todas las barras empiezan en 0 y muestran la n | — |

**¿Qué afirmación sería la más fácil de atacar académicamente?**
*"La educación es la principal vía de salida de la pobreza."* La escolaridad propia se mide al mismo tiempo que el destino, puede ser consecuencia de recursos no observados (causalidad inversa y variables omitidas) y la H003 no encontró evidencia de que su asociación varíe por región. Debe presentarse como **asociación mediadora**, nunca como efecto causal.

## 4. Checklist técnico final (Fase 13)
| Requisito | Estado |
|---|---|
| Ambiente reproducible documentado | ✅ `requirements.txt` con versiones fijas |
| Semillas registradas | ✅ 42 (más una prueba con 10 semillas) |
| Datos originales preservados | ✅ zips en el repo; `data/raw` no se modifica |
| Transformaciones reproducibles | ✅ `src/` + notebooks 01-02 |
| Separación correcta de datos | ✅ 80/20 estratificado + CV de 5 particiones |
| Ausencia de leakage revisada | ✅ |
| Modelos y parámetros registrados | ✅ `src/models/train.py`, `docs/registro_experimentos.md` |
| Métricas reproducibles | ✅ `reports/tables/` |
| Figuras exportadas con títulos y fuentes | ✅ `reports/figures/` |
| Referencias completas | ⚠️ `docs/report/referencias.md` con enlaces; verificación pospuesta por decisión del equipo |
| Limitaciones visibles | ✅ |
| Resultados contradictorios documentados | ✅ (tono de piel, madre, H003) |
| Reporte PDF final | ❌ Pendiente (Fase 12) |
| Auditoría integral | ✅ `docs/audit/` (dictamen APROBADO CON CAMBIOS, correcciones aplicadas) |
| Notebooks o scripts organizados | ✅ |
| README actualizado | ✅ |

## 5. Reproducibilidad
1. Descomprimir los datos en `data/raw/` (ver `docs/base_de_datos.md`).
2. `pip install -r requirements.txt`
3. Ejecutar los notebooks del 01 al 08 en orden.

**Estado de la fase:** APROBADA CON CAMBIOS (auditoría integral aprobada por el equipo el 2026-11-16; ver `docs/audit/`).